# 01 — Inspect the existing Steam snapshot

Start here before model selection. We compare the **full review population** with the overlapping **game-capped text subset**, check data contracts, and inspect time coverage. These cells do not train models or call Steam.

Run `uv sync --locked --extra dev --extra notebooks`, then `uv run --extra notebooks jupyter lab`. Select the project environment as the kernel. Outputs are intentionally empty in Git.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "configs/data.toml").is_file(), "Launch from Game-review/ or notebooks/."

from steam_insights.config import load_config
from steam_insights.data.io import read_csv

config = load_config(PROJECT_ROOT / "configs/data.toml")

## Load CSVs without losing IDs or literal text

The loader handles the UTF-8 BOM, retains Steam IDs as strings, and does not treat a review saying `NA` as a missing value. Do not concatenate full and balanced reviews.

In [ ]:
reviews = read_csv(config.source("reviews"))
balanced = read_csv(config.source("balanced"))
announcements = read_csv(config.source("announcements"))
display(pd.DataFrame({"dataset": ["reviews", "balanced", "announcements"],
                      "rows": [len(reviews), len(balanced), len(announcements)]}))
display(reviews.dtypes.astype(str).to_frame("source_dtype"))

## Validate and generate the exploration report

Inspect errors before preprocessing. Warnings about empty or repeated text do not imply that a recommendation should be dropped. `profile` writes aggregate reports and plots under the configured report directory.

In [ ]:
from steam_insights.data.exploration import profile

report = profile(config)
for validation in report["validation"]:
    print(validation["dataset"], "passed" if validation["valid"] else "FAILED")
    display(pd.DataFrame(validation["issues"]))
print("Report:", config.reports_dir / "README.md")

## Compare populations and recommendation rates

The denominator is shown explicitly. A Steam recommendation is a global vote, not sentiment toward a particular patch change. Different review lengths and per-game caps change the observed distribution.

In [ ]:
comparison = pd.read_csv(config.reports_dir / "population_comparison.csv")
display(comparison[["dataset", "reviews", "games", "negative_votes", "valid_votes", "negative_share", "below_minimum_words", "edited_latest_snapshot"]])
display(pd.read_csv(config.reports_dir / "review_lengths.csv"))

## Inspect missing fields and observed calendar coverage

Unknown playtime/booleans are not zero/false. First and last observed dates may be partial. Daily counts describe the saved sample, not a proof that the API returned every review.

In [ ]:
missing = pd.read_csv(config.reports_dir / "missing_values.csv")
display(missing.loc[missing["missing"].gt(0)].sort_values("missing", ascending=False))
display(pd.read_csv(config.reports_dir / "reviews_by_game.csv"))
display(pd.read_csv(config.reports_dir / "announcements_by_game.csv"))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, filename in zip(axes.flat, ["reviews_by_game.png", "review_lengths.png", "recommendation_comparison.png", "coverage_by_game_day.png"]):
    ax.imshow(plt.imread(config.reports_dir / "figures" / filename))
    ax.set_axis_off()
fig.tight_layout()
plt.show()
plt.close(fig)

## Review announcement flags

Candidate and verification flags were supplied by the earlier heuristic/manual screening. They are not new annotations or supervised fix-status labels. The CSV has flattened text, so original heading scope must be recovered from raw sources later.

In [ ]:
display(pd.Series(report["announcements"]["verification_flags"], name="posts").to_frame())
display(pd.Series(report["announcements"]["branch_hints"], name="posts").to_frame())
print("Heuristic patch candidates:", report["announcements"]["heuristic_patch_candidates"])

## Record decisions before modeling

- Which games have enough observed coverage around a verified patch?
- Which source posts need their original heading/branch context recovered?
- How does the minimum-word choice affect the available evidence?
- Which duplicate-text families and edited reviews need special handling in future evaluation splits?
- What human labels are needed to evaluate relevance, target polarity, and fix status separately?

Write findings in a reviewed document; do not infer causal patch effects from this snapshot.